# Database Explorer

Quick overview of all tables in the database and their most recent data.

In [ ]:
import sys
from pathlib import Path
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / "utils").exists():
    for p in ROOT.parents:
        if (p / "utils").exists():
            ROOT = p
            break
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from utils.helpers import query_db

In [ ]:
# List all tables with row counts and column info
tables = query_db("""
    SELECT
        schemaname AS schema,
        tablename AS table,
        pg_size_pretty(pg_total_relation_size(schemaname || '.' || tablename)) AS size
    FROM pg_tables
    WHERE schemaname NOT IN ('pg_catalog', 'information_schema')
    AND schemaname NOT LIKE '_time%'
    ORDER BY schemaname, tablename
""")
display(tables)

In [ ]:
# Most recent rows for each table
for _, row in tables.iterrows():
    schema, table = row["schema"], row["table"]
    fqn = f"{schema}.{table}"

    # Get columns
    cols = query_db("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = %s AND table_name = %s
        ORDER BY ordinal_position
    """, params=(schema, table))

    col_names = cols["column_name"].tolist()
    col_types = cols["data_type"].tolist()

    # Find a date/timestamp column to sort by
    date_col = None
    for cn, ct in zip(col_names, col_types):
        if ct in ("date", "timestamp without time zone", "timestamp with time zone"):
            date_col = cn
            break

    if date_col:
        recent = query_db(f'SELECT * FROM {fqn} ORDER BY "{date_col}" DESC LIMIT 5')
    else:
        recent = query_db(f"SELECT * FROM {fqn} LIMIT 5")

    display(Markdown(f"### `{fqn}`"))
    col_summary = ", ".join(f"`{n}` ({t})" for n, t in zip(col_names, col_types))
    display(Markdown(f"Columns: {col_summary}"))
    display(recent)
    print()